In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
from crewai import Agent, Task, Crew, Process

In [ ]:
load_dotenv(override=True)

In [ ]:
# 2. Hard-kill all outgoing CrewAI telemetry and tracing pipelines
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["CREWAI_TRACING_ENABLED"] = "false"

In [ ]:
openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

if openai_api_key:
    print(f"OpenAI API key eists and begins with -  {openai_api_key[:8]}")
else:
    prit("OpenAI key not set")
if google_api_key:
    print(f"Google API key eists and begins with -  {google_api_key[:8]}")
else:
    prit("Google API key not set")
if groq_api_key:
    print(f"Groq API key eists and begins with -  {groq_api_key[:8]}")
else:
    prit("Groq API key not set")

#### Cell 1: Installation of Required Frameworks
---
We need crewai and its tools for building the multi-agent system, langchain-google-genai to interface smoothly with Google's Gemini models, and python-docx to programmatic-ally compile the final text into a Microsoft Word document.

In [ ]:
# !pip install crewai crewai_tools langchain-google-genai python-docx

In [ ]:
# Cell 2: Imports and Environment Setup
# =====================================
import os
import getpass
from crewai import Agent, Task, Crew, Process
from langchain_google_genai import ChatGoogleGenerativeAI
from crewai_tools import FileWriterTool, FileReadTool

In [ ]:
from langchain_community.callbacks.manager import get_openai_callback

In [ ]:
# Setup API Key securely without hardcoding it in the script
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

In [ ]:
# Initialize our default LLM. We are using gemini-3.5-flash for its efficiency
# and highly responsive structural execution.

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    verbose=True,
    temperature=0.2,
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

print("LLM initialized and environment configurations successfully loaded.")

#### Troubleshooting CrewAI ValidationError
---
1.  The issue stems from passing a Google Generative AI LLM object directly into the llm field of a CrewAI `Agent`. The underlying validation library, Pydantic, expects either a plain string name of an LLM or an instance that directly inherits from LangChain's `BaseLLM`.

2.  When using Google models through LangChain (`ChatGoogleGenerativeAI`), the instantiated object does not natively align with what CrewAI's `Agent` schema requires by default. CrewAI expects the native `llm` parameter to follow its strict integration paths.

3.  To resolve this error, you need to correctly bridge your Google LLM instance with the `Agent` class. CrewAI allows you to pass a LangChain-wrapped chat model directly, but you must ensure you have imported and configured `ChatGoogleGenerativeAI` properly, or explicitly pass the string name if you are relying on CrewAI's internal LLM manager.

4.   Or use  the `BaseLLM` from crewai library:

In [ ]:
from crewai import LLM

# Wrap your Gemini model properly using CrewAI's LLM wrapper
llm = LLM(
    model="gemini/gemini-3.5-flash",  # Use provider/model format
    temperature=0.2,
    api_key=os.getenv("GOOGLE_API_KEY")  # Or let it read from os.environ["GEMINI_API_KEY"]
)

print("LLM initialized and environment configurations successfully loaded.")

# Now your agents will accept this 'llm' variable without any validation errors!

In [ ]:
# Cell 3: Instantiating Shared File Tools
# =======================================
# To fulfill your requirement of injecting humanization rules directly from an external asset,
# we use the FileReadTool to pull data from skills/humanizer/SKILL.md.
# The FileWriterTool will be handed to the document generation agent to compile the final file.

file_reader = FileReadTool(file_path="skills/humanizer/SKILL.md")
file_writer = FileWriterTool()

print("File tools initialized.")

In [ ]:
# Cell 4: Agent Definitions
# =========================
# We are creating individual specialized agents for each distinct task. This design decouples
# responsibilities so that no single agent gets overwhelmed or hallucinating details.

topic = "Environmental Security in India" 
context = "AI infra requirements, climate change, rainfed agri, groundwater extraction, and food security"

# 1. Introduction Agent
intro_agent = Agent(
    role="Strategic Framing Specialist",
    goal=f"Draft an engaging, impactful introduction setting the context of {topic}. Use a highly relevant quote to begin with and then seamlessly follow up with the introduction narrative.",
    backstory=f"You are an elite academic essayist who hooks readers by introducing the stark realities, history, and modern relevance of {topic} within the context of {context}.",
    llm=llm,
    max_iter=3,
    verbose=True
)

# 2. Core Argument Agent
core_argument_agent = Agent(
    role="Policy and Strategic Analyst",
    goal=f"Establish the primary hypothesis and core arguments concerning the fundamental vulnerabilities or opportunities of {topic}.",
    backstory=f"You specialize in systemic policy and administrative strategy, explaining how structural issues within {topic} translate into broader societal or institutional risks.",
    llm=llm,
    max_iter=3,
    verbose=True
)

# 3. Multi-Dimensional Analyst Agent
dimension_analyst_agent = Agent(
    role="Socio-Economic & Geopolitical Researcher",
    goal=f"Examine {topic} across multiple dimensions: economic impacts, social shifts, and geopolitical or industry-wide stresses.",
    backstory=f"You are an expert sociologist and macroeconomist tracking systemic resource allocation, regional/industry instability, and human impact related to {topic}.",
    llm=llm,
    max_iter=3,
    verbose=True
)

# 4. Facts and Case Studies Finder Agent
evidence_collector_agent = Agent(
    role="Lead Investigator and Case Studies Archivist",
    goal=f"Provide verified recent examples, specific data points, and concrete case studies relevant to {topic}.",
    backstory=f"You compile critical evidentiary facts, historical precedents, and real-world incidents that perfectly illustrate the core challenges of {topic}.",
    llm=llm,
    max_iter=3,
    verbose=True
)

# 5. Counter Arguments Agent
counter_argument_agent = Agent(
    role="Critical Dialectic Strategist",
    goal=f"Address counterpoints, economic trade-offs, and alternative viewpoints fairly to balance the essay on {topic}.",
    backstory="You play devil's advocate, evaluating opposing arguments, institutional defenses, and competing priorities to preemptively solidify the report's credibility.",
    llm=llm,
    max_iter=3,
    verbose=True
)

# 6. Synthesis and Conclusion Agent
conclusion_agent = Agent(
    role="Chief Synthesizer and Visionary Policy Maker",
    goal=f"Weave overarching conclusions and visionary strategic recommendations regarding {topic}.",
    backstory="You look ahead, crafting compelling conclusions and sustainable roadmaps tailored to administrative or organizational adoption.",
    llm=llm,
    max_iter=3,
    verbose=True
)

# 7. Independent Auditor (Fact-Checker and Humanizer) Agent
auditor_agent = Agent(
    role="Independent Fact-Checker and Chief Tonal Editor",
    goal=f"Independently review all drafted sections on {topic} to verify claims and heavily humanize the language based on strict stylistic guidelines.",
    backstory="You strip away corporate jargon, overly structured AI catchphrases, and ensure rhythmic, organic human writing styles while rigorously cross-checking structural claims.",
    tools=[file_reader],
    llm=llm,
    verbose=True
)

# 8. Report Assembler Agent
assembler_agent = Agent(
    role="Executive Editor and Manuscript Compiler",
    goal=f"Synthesize, organize, and compile all verified, humanized sub-sections of {topic} into one single cohesive master document text.",
    backstory="You ensure flawless transitions between sections, smooth stylistic continuity, and flawless structural flow across the entire text.",
    llm=llm,
    verbose=True
)

# 9. Word Document Generator Agent
doc_generator_agent = Agent(
    role="Document Automation Architect",
    goal="Format the complete manuscript text and write it into a cleanly structured Microsoft Word Document layout.",
    backstory="You are an operational assistant with strict attention to typographical layouts, headers, and file output protocols.",
    tools=[file_writer],
    llm=llm,
    verbose=True
)

print(f"All 9 specialized agents have been successfully generalized for the topic: '{topic}'.")

In [ ]:
# Cell 5: Tasks Definitions
# =========================
# Tasks are executed sequentially. We pass outputs along the pipeline, ensuring that 
# the Auditor validates information, the Compiler merges it, and the Generator saves it.

# Mapping to the standardized variables established in Cell 4
topic_context = f"{topic} in the context of {context}"

intro_word_limit = 150
core_word_limit = 100
arguments_word_limit = 450
facts_word_limit = 200
counter_word_limit = 150
conclusion_word_limit = 150

report_word_limit = 1200

# Drafting Tasks
task_intro = Task(
    description=f"Draft a compelling introductory section for a report on: {topic_context}. Begin with a highly relevant quote and seamlessly transition into the narrative framework. Strict limitation: Do not exceed {intro_word_limit} words.",
    expected_output=f"A raw text introductory framework starting with an impactful quote. Length must strictly be under {intro_word_limit} words.",
    agent=intro_agent
)

task_core = Task(
    description=f"Draft the core thesis and macro arguments outlining the existential threats and primary structural vulnerabilities of {topic_context}. Strict limitation: Do not exceed {core_word_limit} words.",
    expected_output=f"A deep narrative text establishing core systemic arguments, capped strictly at {core_word_limit} words.",
    agent=core_argument_agent
)

task_dimensions = Task(
    description=f"Analyze the multidimensional impacts (Socio-economic shifts, regional/industry stress, rainfed agriculture, and groundwater extraction) regarding: {topic_context}. Strict limitation: Do not exceed {arguments_word_limit} words.",
    expected_output=f"A comprehensive multi-dimensional analytical section, restricted to a maximum of {arguments_word_limit} words.",
    agent=dimension_analyst_agent
)

task_evidence = Task(
    description=f"Provide a meticulous list of recent real-world case studies, data points, and evidentiary historical incidents related to: {topic_context}. Focus on concrete facts. Strict limitation: Do not exceed {facts_word_limit} words.",
    expected_output=f"An evidence-rich dossier containing targeted historical and recent data points across India, restricted to {facts_word_limit} words.",
    agent=evidence_collector_agent
)

task_counter = Task(
    description=f"Draft a robust counter-argument section balancing competing priorities, structural defenses, and trade-offs (e.g., tech infrastructure demand vs. resource preservation) regarding: {topic_context}. Strict limitation: Do not exceed {counter_word_limit} words.",
    expected_output=f"A balanced, dialectical section evaluating development trade-offs, strictly under {counter_word_limit} words.",
    agent=counter_argument_agent
)

task_conclusion = Task(
    description=f"Draft the forward-looking sustainable roadmap, visionary recommendations, and a strong policy conclusion tailored for organizational adoption regarding: {topic_context}. Strict limitation: Do not exceed {conclusion_word_limit} words.",
    expected_output=f"An impactful conclusion and strategic policy recommendations section, limited to {conclusion_word_limit} words.",
    agent=conclusion_agent
)

# Verification, Fact-Checking, and Humanization Task
task_audit = Task(
    description=(
        f"Gather the outputs from ALL the previous drafting tasks (Introduction, Core, Dimensions, Evidence, Counter-arguments, and Conclusion) focused on {topic}. "
        "1. Read the humanization instructions located in 'skills/humanizer/SKILL.md' using your FileRead tool. "
        "2. Review each section independently to ensure facts are sound and realistic. "
        "3. Rewrite the text completely removing common AI patterns, monotonous rhythms, and robotic phrasing according to the markdown instructions. "
        "Maintain the exact factual integrity and preserve the individual section word limits established previously."
    ),
    expected_output="An audited, fact-checked, and deeply humanized version of all text sections separated by clear structural markers.",
    agent=auditor_agent
)

# Composition Task
task_assemble = Task(
    description=f"Take the audited and humanized sections from the Auditor and stitch them together into a unified, seamless executive report on {topic}. Smooth out structural transitions so it reads like a single-author continuous document. Strict limitation: The entire compiled manuscript must not exceed {report_word_limit} words.",
    expected_output=f"A complete, unified, continuous master manuscript draft strictly under {report_word_limit} words.",
    agent=assembler_agent
)

# Automated File Generation Task
task_generate_doc = Task(
    description="Take the continuous master manuscript document from the Assembler. Write it directly into a clean markdown or raw text structure using your file writing tool to 'india_climate_security_report.txt'. Ensure all structural headers are preserved.",
    expected_output="Confirmation message indicating that the complete report has been written successfully to disk.",
    agent=doc_generator_agent
)

print("Sequential operational workflow mapped completely with integrated constraints.")

In [ ]:
# Cell 6: Executing the Crew Framework
# ===================================
# We coordinate the orchestration explicitly via a sequential processing pipeline.

governance_crew = Crew(
    agents=[
        intro_agent, core_argument_agent, dimension_analyst_agent, 
        evidence_collector_agent, counter_argument_agent, conclusion_agent,
        auditor_agent, assembler_agent, doc_generator_agent
    ],
    tasks=[
        task_intro, task_core, task_dimensions, 
        task_evidence, task_counter, task_conclusion,
        task_audit, task_assemble, task_generate_doc
    ],
    max_tokens=100000, 
    process=Process.sequential,
    verbose=True
)

In [ ]:
print("Starting execution pipeline. This may take a moment as agents cross-examine data components sequentially...")

final_summary = governance_crew.kickoff()

#### RuntimeError: Environmental Conflict Error
---
1.  This error boils down to a classic environmental conflict: you are trying to run a synchronous operation (`crew.kickoff()`) inside an environment that is already running an active asynchronous event loop (your Jupyter Notebook kernel).

2.  In standard Python scripts, `crew.kickoff()` handles its internal operations synchronously without issue. However, Jupyter Notebooks run on top of an `asyncio` event loop by default. When CrewAI attempts to trigger synchronous invocations within that active backend loop, the system hits a protective wall and throws:
`RuntimeError: Agent execution was invoked synchronously from within a running event loop.`

3.  The minor telemetry network warning at the bottom (`telemetry.crewai.com`) is just a secondary symptom caused by the main crash interrupting the telemetry data handoff.

4.  **The Fix: Switch to Async Execution**
To resolve this, you need to use CrewAI’s built-in asynchronous execution methods and use Python's `await` keyword so it coordinates smoothly with your Jupyter Notebook's event loop.

#### AuthenticationError
---
##### `ERROR:crewai.flow.runtime:Error executing listener call_llm_and_parse: ...`
---
1.  You are using CrewAI's new **Flows** feature (`crewai.flow.runtime`) alongside your `governance_crew`.

2.  While you successfully mapped your `llm=llm` parameter to all of your individual Agents, the overarching CrewAI **Flow** instance itself has its own independent internal router method (`call_llm_and_parse`) used for state management. <u>Because that Flow is not explicitly told to use Gemini, it defaults back to OpenAI</u>, picks up your Google API key from os.environ, and attempts to validate it against OpenAI’s authentication endpoint, throwing the `401 Unauthorized exception`.

3.  Here is the exact code modification needed to force the **Flow** architecture to target Gemini.

4.  **The Fix**: Bind the LLM directly to your Flow Class
    *   In CrewAI Flows, you must declare a class-level or state-level llm property right inside your **Flow** structure so that internal methods like call_llm_and_parse know which engine to use.
    *   Update your Flow definition structure to mirror this pattern:

In [ ]:
from crewai.flow.flow import Flow, listen, start

# 1. Re-verify your base configuration instantiation
llm = LLM(
    model="gemini/gemini-3.5-flash",  # Use provider/model format
    temperature=0.2,
    api_key=os.getenv("GOOGLE_API_KEY")  # Or let it read from os.environ["GEMINI_API_KEY"]
)

# 2. Bind the LLM inside your Flow architecture definition
class GovernanceFlow(Flow):
    # CRITICAL: This class-level attribute overrides the default OpenAI fallback
    llm = llm 

    @start()
    def start_pipeline(self):
        print("Starting execution pipeline...")
        # Kickoff the crew (making sure agents have llm=llm bound too)
        # return governance_crew.kickoff()
        pass

    @listen(start_pipeline)
    def process_results(self, crew_output):
        # Any internal flow calling happens here using Gemini llm now!
        pass

flow_instance = GovernanceFlow()

#### NameResolutionError
--- 
##### `NameResolutionError("HTTPSConnection(host='telemetry.crewai.com', port=4319): Failed to resolve 'telemetry.crewai.com' ([Errno 11001] getaddrinfo failed)"))`
---

1.  That error message means **CrewAI is trying to call home to its telemetry servers, but your system's network cannot reach or resolve their web address** (`telemetry.crewai.com`).

2.  This frequently happens if you are behind a strict corporate firewall, a VPN, have an unstable internet connection, **or if CrewAI’s telemetry endpoint is experiencing a temporary outage**.

3.   While this error looks scary, it is completely non-blocking for your actual local AI logic. The core agent execution runs on your machine and communicates with Google AI Studio just fine; it only crashes at the very end when trying to upload anonymous usage statistics to CrewAI.

4.  The cleanest and most reliable **fix is to completely disable telemetry** in your project so CrewAI stops trying to make this external network call entirely.

5.  <u>**Step 1: Disable Telemetry in Your `.env File`**</u>
    *   Open the `.env` file at the root of your project workspace and append this flag to the bottom:
        *   `OTEL_SDK_DISABLED=true`
        *   `CREWAI_TRACING_ENABLED=false`

6.  <u>**Step 2: Set the Environment Flag Directly in Your Notebook**</u>
    *   To make absolutely sure your running Jupyter notebook picks up the instruction immediately without needing to hunt down file path bindings, add these lines to the very top code cell of your notebook (right where you handle `load_dotenv()`):
        *   ```<br>
            import os
            from dotenv import load_dotenv

            # 1. Load your local environmental variables
            load_dotenv(override=True)

            # 2. Hard-kill all outgoing CrewAI telemetry and tracing pipelines
            os.environ["OTEL_SDK_DISABLED"] = "true"
            os.environ["CREWAI_TRACING_ENABLED"] = "false"
            ```

7.  <u>**Step 3: Run via the Command Line Interface**</u>
If you ever execute your agents via a standard terminal command or use a structured project format later, you can also force-disable it by running this command once in your terminal:
    *   `crewai traces disable`

8.  <u>**Why this fixes it:**</u>
CrewAI uses an OpenTelemetry (`OTEL`) software development kit background layer to track task durations and execution structures. By injecting` OTEL_SDK_DISABLED=true`, you tell the backend package to completely skip initializing that network thread.
    *   Once you add that flag, restart your notebook kernel, and execute your cells, your `governance_crew` will run cleanly to completion without throwing any trailing connectivity exceptions!
   

In [ ]:
print("Starting execution pipeline. This may take a moment as agents cross-examine data components sequentially...")

final_summary = await governance_crew.kickoff_async()

print("\n=== SYSTEM PIPELINE COMPLETE ===")
print(final_summary)

In [ ]:
Markdown(final_summary.raw)

In [ ]:
from langchain_community.callbacks.manager import get_openai_callback

with get_openai_callback() as cb:
    print("🚀 Initiating Agentic Workflow...")
    result = my_crew.kickoff()
    
    print("\n" + "="*40)
    print("📊 TOKEN MONITORING METRICS")
    print("="*40)
    print(f"Total Tokens Used:      {cb.total_tokens}")
    print(f"Prompt (Input) Tokens:  {cb.prompt_tokens}")
    print(f"Completion (Output):    {cb.completion_tokens}")
    print(f"Total Execution Cost:   ${cb.total_cost:.6f}")
    print("="*40)